In [2]:
from datetime import datetime, timedelta, date
import yfinance as yf

def get_stock_price_date(ticker, datestr, pricetype = 'Close', n_try=0):
    date = datetime.strptime(datestr, '%Y-%m-%d')
    data = yf.download(ticker, start = date, end = date + timedelta(days=1))
    if data[pricetype].values.size == 0:
      if n_try > 5:
        return None
      return get_stock_price_date(ticker, datetime.strftime(date + timedelta(days=1), '%Y-%m-%d'), pricetype, n_try+1)  # If no data, try the next day
    else:
      return {'datastr': datestr, 'price': float(data[pricetype].values[0][0])}

def get_stock_info_daterange(ticker, start_date, end_date):
  stock = yf.Ticker(ticker)
  data = stock.history(start=start_date, end=end_date).reset_index()
  return data.to_json()

In [3]:
get_stock_price_date('HOOD', '2025-11-01')

$HOOD: possibly delisted; no price data found  (1d 2025-11-01 00:00:00 -> 2025-11-02 00:00:00)
[*********************100%***********************]  1 of 1 completed

1 Failed download:
['HOOD']: possibly delisted; no price data found  (1d 2025-11-01 00:00:00 -> 2025-11-02 00:00:00)
$HOOD: possibly delisted; no price data found  (1d 2025-11-02 00:00:00 -> 2025-11-03 00:00:00)
[*********************100%***********************]  1 of 1 completed

1 Failed download:
['HOOD']: possibly delisted; no price data found  (1d 2025-11-02 00:00:00 -> 2025-11-03 00:00:00)
[*********************100%***********************]  1 of 1 completed


{'datastr': '2025-11-03', 'price': 147.0800018310547}

In [4]:
res = get_stock_info_daterange('HOOD', '2025-11-01', '2025-12-01')

C:\Users\RyanJ\AppData\Local\Temp\ipykernel_43228\3275510878.py:17: Pandas4Warning: The default 'epoch' date format is deprecated and will be removed in a future version, please use 'iso' date format instead.
  return data.to_json()


In [5]:
res

'{"Date":{"0":1762146000000,"1":1762232400000,"2":1762318800000,"3":1762405200000,"4":1762491600000,"5":1762750800000,"6":1762837200000,"7":1762923600000,"8":1763010000000,"9":1763096400000,"10":1763355600000,"11":1763442000000,"12":1763528400000,"13":1763614800000,"14":1763701200000,"15":1763960400000,"16":1764046800000,"17":1764133200000,"18":1764306000000},"Open":{"0":148.7899932861,"1":139.8699951172,"2":139.6349945068,"3":140.0,"4":123.7659988403,"5":135.4499969482,"6":133.9900054932,"7":133.3300018311,"8":129.8699951172,"9":115.1500015259,"10":120.4950027466,"11":113.9100036621,"12":114.7399978638,"13":121.6299972534,"14":107.8399963379,"15":109.9150009155,"16":112.4649963379,"17":121.3300018311,"18":128.0749969482},"High":{"0":149.4120025635,"1":144.3999938965,"2":144.7700042725,"3":140.1600036621,"4":132.1999969482,"5":138.9900054932,"6":134.6670074463,"7":133.8399963379,"8":130.1499938965,"9":126.5,"10":120.9800033569,"11":116.9710006714,"12":119.1999969482,"13":122.6299972534

# define functions

In [6]:
get_stock_price_date_function = {
    "name": "get_stock_price_date",
    "description": "Get stock price at a specific date with specified price type.",
    "parameters": {
        "type": "object",
        "properties": {
            "ticker": {
                "type": "string",
                "description": "Ticker symbol of the stock (e.g., 'HOOD')",
            },
            "datestr": {
                "type": "string",
                "description": "Date of which the price is returned (e.g., '2025-04-09')",
            },
            "pricetype": {
                "type": "string",
                "description": "The type of the price, can only be chosen from 'Open', 'Close', 'High', 'Low'. Default is 'Close'.",
            },
            "n_try": {
                "type": "integer",
                "description": "Number of tries made if the date falls to a holiday or weekend when no stock data are available. Please do not pass this argument as it defaults to zero.",
            },
        },
        "required": ["ticker", "datestr"],
    },
}

get_stock_info_daterange_function = {
    "name": "get_stock_info_daterange",
    "description": "Get stock history information within a given date range, including open, close, high, low prices, dividends and splits.",
    "parameters": {
        "type": "object",
        "properties": {
            "ticker": {
                "type": "string",
                "description": "Ticker symbol of the stock (e.g., 'HOOD')",
            },
            "start_date": {
                "type": "string",
                "description": "Start date string of which the stock history needs to be retrieved (e.g., '2025-04-09')",
            },
            "end_date": {
                "type": "string",
                "description": "End date string of which the stock history needs to be retrieved (e.g., '2025-05-01')",
            },
        },
        "required": ["ticker", "start_date", "end_date"],
    },
}

In [13]:
from credentials import gemini_api_key
import os
os.environ['gemini_api_key'] = gemini_api_key
model_name = 'gemini-3-flash-preview'

from google import genai
from google.genai import types
import time

class gemini:
  def __init__(self, model_name, api_key, tools=None, system_instruction=None):
    self.model_name = model_name
    self.client = genai.Client(api_key = api_key)
    self.system_instruction = system_instruction
    if tools:
      self.tools = types.Tool(function_declarations=tools)
      self.config = types.GenerateContentConfig(tools=[self.tools], system_instruction = self.system_instruction)
    else:
      self.config = types.GenerateContentConfig(system_instruction = self.system_instruction)
    self.contents = []

  def call(self, prompt=None, delay = 60, max_try=5):
    if prompt:
      self.contents.append(
          types.Content(
              role="user", parts=[types.Part(text=prompt)]
          )
      )
    n_try = 1
    while n_try < max_try:
      try:
        response = self.client.models.generate_content(
            model=self.model_name,
            config=self.config,
            contents=self.contents,
        )
        break
      except Exception as e:
        print(f"Error: {e}.")
        print(f"Retrying in {delay} seconds...")
        time.sleep(delay)
        n_try += 1
        delay *= 2
        if n_try == max_try:
          print("Failed to generate content after multiple retries.")
          return None

    content = response.candidates[0].content
    self.contents.append(content)

    for p in content.parts:
      if p.function_call:
        print(f"Function to call: {p.function_call.name}")
        print(f"Arguments: {p.function_call.args}")
        result = eval(f'{p.function_call.name}(**{p.function_call.args})')
        print(result)

        function_response_part = types.Part.from_function_response(
          name=p.function_call.name,
          response={"result": result},
        )
        self.contents.append(types.Content(role="user", parts=[function_response_part])) # Append the function response
      else:
        print("No function call found in the response.")
        print(p.text)
    return response


# Example 1

In [17]:
g = gemini('gemini-3-flash-preview', os.environ['gemini_api_key'], tools=[get_stock_price_date_function, get_stock_info_daterange_function])

In [18]:
response = g.call("How much did Robinhood increase from Nov 1st last year to today?")

Function to call: get_stock_price_date
Arguments: {'datestr': '2023-11-01', 'ticker': 'HOOD'}


[*********************100%***********************]  1 of 1 completed


{'datastr': '2023-11-01', 'price': 9.079999923706055}
Function to call: get_stock_price_date
Arguments: {'datestr': '2024-05-15', 'ticker': 'HOOD'}


[*********************100%***********************]  1 of 1 completed

{'datastr': '2024-05-15', 'price': 18.520000457763672}


In [19]:
r2 = g.call()

No function call found in the response.
Robinhood's stock (HOOD) increased by **$9.44** from November 1, 2023, to today (May 15, 2024).

*   **Price on Nov 1, 2023:** $9.08
*   **Price on May 15, 2024:** $18.52
*   **Total Increase:** ~103.96%


# Example 2

In [26]:
date_today = date.today().strftime('%Y-%m-%d')

sys_prompt = f"""Today is {date_today}, you are a helpful assistant to answer financial questions.
"""
g = gemini('gemini-3-flash-preview', os.environ['gemini_api_key'], tools=[get_stock_price_date_function, get_stock_info_daterange_function], system_instruction=sys_prompt)

In [27]:
r1 = g.call("How much did Robinhood increase from Nov 1st last year to today?")

$HOOD: possibly delisted; no price data found  (1d 2025-11-01 00:00:00 -> 2025-11-02 00:00:00)
[*********************100%***********************]  1 of 1 completed

1 Failed download:
['HOOD']: possibly delisted; no price data found  (1d 2025-11-01 00:00:00 -> 2025-11-02 00:00:00)
$HOOD: possibly delisted; no price data found  (1d 2025-11-02 00:00:00 -> 2025-11-03 00:00:00)
[*********************100%***********************]  1 of 1 completed

1 Failed download:
['HOOD']: possibly delisted; no price data found  (1d 2025-11-02 00:00:00 -> 2025-11-03 00:00:00)
[*********************100%***********************]  1 of 1 completed

Function to call: get_stock_price_date
Arguments: {'datestr': '2025-11-01', 'ticker': 'HOOD'}
{'datastr': '2025-11-03', 'price': 147.0800018310547}
Function to call: get_stock_price_date
Arguments: {'datestr': '2026-03-03', 'ticker': 'HOOD'}



[*********************100%***********************]  1 of 1 completed

{'datastr': '2026-03-03', 'price': 76.06999969482422}


In [28]:
r2 = g.call()

Function to call: get_stock_info_daterange
Arguments: {'ticker': 'HOOD', 'start_date': '2025-11-01', 'end_date': '2026-03-03'}
{"Date":{"0":1762146000000,"1":1762232400000,"2":1762318800000,"3":1762405200000,"4":1762491600000,"5":1762750800000,"6":1762837200000,"7":1762923600000,"8":1763010000000,"9":1763096400000,"10":1763355600000,"11":1763442000000,"12":1763528400000,"13":1763614800000,"14":1763701200000,"15":1763960400000,"16":1764046800000,"17":1764133200000,"18":1764306000000,"19":1764565200000,"20":1764651600000,"21":1764738000000,"22":1764824400000,"23":1764910800000,"24":1765170000000,"25":1765256400000,"26":1765342800000,"27":1765429200000,"28":1765515600000,"29":1765774800000,"30":1765861200000,"31":1765947600000,"32":1766034000000,"33":1766120400000,"34":1766379600000,"35":1766466000000,"36":1766552400000,"37":1766725200000,"38":1766984400000,"39":1767070800000,"40":1767157200000,"41":1767330000000,"42":1767589200000,"43":1767675600000,"44":1767762000000,"45":1767848400000,

C:\Users\RyanJ\AppData\Local\Temp\ipykernel_43228\3275510878.py:17: Pandas4Warning: The default 'epoch' date format is deprecated and will be removed in a future version, please use 'iso' date format instead.
  return data.to_json()


In [29]:
r3 = g.call()

No function call found in the response.
From November 1st, 2025, to today (March 3, 2026), the price of Robinhood (HOOD) stock has actually **decreased** rather than increased.

Based on the market data:
*   **Price on November 1, 2025:** Approximately **$147.08** (observed on the first following trading day, November 3rd).
*   **Current Price (March 3, 2026):** Approximately **$76.07**.

### Performance Summary:
*   **Total Change:** -$71.01 per share
*   **Percentage Change:** **-48.28%**

There were no stock splits or dividends recorded during this specific period that would have artificially impacted these price points. The stock experienced a significant downward trend over the last four months, losing nearly half of its value from its November high.


# Example 3

In [31]:
sys_prompt = f"""Today is {date_today}, you are a helpful assistant to answer financial questions.
"""
g = gemini('gemini-3-flash-preview', os.environ['gemini_api_key'], tools=[get_stock_price_date_function, get_stock_info_daterange_function], system_instruction=sys_prompt)
r1 = g.call("Should I invest gold or Google last year to get the maximum return?")

Function to call: get_stock_price_date
Arguments: {'ticker': 'GOOGL', 'datestr': '2025-01-02', 'pricetype': 'Close'}


[*********************100%***********************]  1 of 1 completed


{'datastr': '2025-01-02', 'price': 188.6909942626953}
Function to call: get_stock_price_date
Arguments: {'ticker': 'GOOGL', 'datestr': '2025-12-31', 'pricetype': 'Close'}


[*********************100%***********************]  1 of 1 completed


{'datastr': '2025-12-31', 'price': 313.0}
Function to call: get_stock_price_date
Arguments: {'ticker': 'GLD', 'datestr': '2025-01-02', 'pricetype': 'Close'}


[*********************100%***********************]  1 of 1 completed


{'datastr': '2025-01-02', 'price': 245.4199981689453}
Function to call: get_stock_price_date
Arguments: {'pricetype': 'Close', 'datestr': '2025-12-31', 'ticker': 'GLD'}


[*********************100%***********************]  1 of 1 completed

{'datastr': '2025-12-31', 'price': 396.30999755859375}


In [32]:
r2 = g.call()

Function to call: get_stock_info_daterange
Arguments: {'start_date': '2025-01-01', 'end_date': '2025-12-31', 'ticker': 'GOOGL'}


C:\Users\RyanJ\AppData\Local\Temp\ipykernel_43228\3275510878.py:17: Pandas4Warning: The default 'epoch' date format is deprecated and will be removed in a future version, please use 'iso' date format instead.
  return data.to_json()


{"Date":{"0":1735794000000,"1":1735880400000,"2":1736139600000,"3":1736226000000,"4":1736312400000,"5":1736485200000,"6":1736744400000,"7":1736830800000,"8":1736917200000,"9":1737003600000,"10":1737090000000,"11":1737435600000,"12":1737522000000,"13":1737608400000,"14":1737694800000,"15":1737954000000,"16":1738040400000,"17":1738126800000,"18":1738213200000,"19":1738299600000,"20":1738558800000,"21":1738645200000,"22":1738731600000,"23":1738818000000,"24":1738904400000,"25":1739163600000,"26":1739250000000,"27":1739336400000,"28":1739422800000,"29":1739509200000,"30":1739854800000,"31":1739941200000,"32":1740027600000,"33":1740114000000,"34":1740373200000,"35":1740459600000,"36":1740546000000,"37":1740632400000,"38":1740718800000,"39":1740978000000,"40":1741064400000,"41":1741150800000,"42":1741237200000,"43":1741323600000,"44":1741579200000,"45":1741665600000,"46":1741752000000,"47":1741838400000,"48":1741924800000,"49":1742184000000,"50":1742270400000,"51":1742356800000,"52":17424432

C:\Users\RyanJ\AppData\Local\Temp\ipykernel_43228\3275510878.py:17: Pandas4Warning: The default 'epoch' date format is deprecated and will be removed in a future version, please use 'iso' date format instead.
  return data.to_json()


In [33]:
r3 = g.call()

No function call found in the response.
To determine whether you should have invested in gold or Google (Alphabet Inc.) last year (2025), we can compare their performance based on their total returns from the beginning to the end of the year.

For this comparison, we will use **Alphabet Inc. (GOOGL)** as the representative for Google and the **SPDR Gold Shares (GLD)** ETF as the representative for an investment in gold.

### Performance Comparison (2025)

| Investment | Price (Jan 2, 2025) | Price (Dec 31, 2025) | Dividends Paid | Total Return (%) |
| :--- | :--- | :--- | :--- | :--- |
| **Google (GOOGL)** | $188.69 | $313.85 | $0.83 | **~66.77%** |
| **Gold (GLD)** | $245.42 | $398.89 | $0.00 | **~62.53%** |

### Analysis
*   **Google (Alphabet):** Alphabet saw significant growth in 2025, with its stock price increasing by approximately 66.3% over the year. When including the quarterly dividends of roughly $0.20–$0.21 per share, the total return reached approximately **66.77%**.
*   *

# Example 4

In [38]:
sys_prompt = f"""Today is {date_today}, you are a helpful assistant to answer financial questions.
"""
g = gemini('gemini-3-flash-preview', os.environ['gemini_api_key'], tools=[get_stock_price_date_function, get_stock_info_daterange_function], system_instruction=sys_prompt)
r1 = g.call("What are the average closing price for gold this year? Is it higher than the gold price today? If so, by how much?")

Function to call: get_stock_info_daterange
Arguments: {'ticker': 'GC=F', 'end_date': '2026-03-03', 'start_date': '2026-01-01'}
{"Date":{"0":1767330000000,"1":1767589200000,"2":1767675600000,"3":1767762000000,"4":1767848400000,"5":1767934800000,"6":1768194000000,"7":1768280400000,"8":1768366800000,"9":1768453200000,"10":1768539600000,"11":1768885200000,"12":1768971600000,"13":1769058000000,"14":1769144400000,"15":1769403600000,"16":1769490000000,"17":1769576400000,"18":1769662800000,"19":1769749200000,"20":1770008400000,"21":1770094800000,"22":1770181200000,"23":1770267600000,"24":1770354000000,"25":1770613200000,"26":1770699600000,"27":1770786000000,"28":1770872400000,"29":1770958800000,"30":1771304400000,"31":1771390800000,"32":1771477200000,"33":1771563600000,"34":1771822800000,"35":1771909200000,"36":1771995600000,"37":1772082000000,"38":1772168400000,"39":1772427600000},"Open":{"0":4350.6000976562,"1":4386.7001953125,"2":4449.7001953125,"3":4450.0,"4":4460.2001953125,"5":4473.0,"6"

C:\Users\RyanJ\AppData\Local\Temp\ipykernel_43228\3275510878.py:17: Pandas4Warning: The default 'epoch' date format is deprecated and will be removed in a future version, please use 'iso' date format instead.
  return data.to_json()


In [39]:
r2 = g.call()

Function to call: get_stock_price_date
Arguments: {'datestr': '2026-03-03', 'ticker': 'GC=F'}


[*********************100%***********************]  1 of 1 completed

{'datastr': '2026-03-03', 'price': 5130.2998046875}


In [40]:
r3 = g.call()

No function call found in the response.
The average closing price for gold (GC=F) so far in 2026 is approximately **$4,884.19**.

As of today, March 3, 2026, the closing price for gold is **$5,130.30**.

Comparing the two:
*   **Is the average higher than today's price?** No.
*   **By how much?** Today's price is actually higher than the yearly average by **$246.11**.


### without tool use

In [41]:
g = gemini('gemini-3-flash-preview', os.environ['gemini_api_key'], system_instruction=sys_prompt)
r1 = g.call("What are the average closing price for gold this year? Is it higher than the gold price today? If so, by how much?")

No function call found in the response.
As an AI, I do not have access to a real-time financial ticker for the year 2026. My internal database includes historical market data up through late 2024 and early 2025. To get the exact figures for today, March 3, 2026, you would need to consult a live financial terminal such as Bloomberg, Reuters, or a dedicated gold market site like Kitco.

However, I can provide you with the context of where gold prices were trending leading into this period and show you how to calculate that difference.

### Context from Recent Trends (Late 2024/Early 2025)
In the period leading up to 2025, gold experienced significant volatility and reached record highs:
*   **Late 2024 Peak:** Gold prices surged past **$2,700 per ounce** due to geopolitical instability, central bank purchases, and shifts in U.S. interest rate expectations.
*   **Market Sentiment:** Analysts at that time predicted that if inflation remained sticky or global tensions persisted, gold could 